# PC3 - Deep Learning - Notebook 3
### Mucha Alvarez Diego Fabrizio - u202317069

# 03 · Transfer Learning

Usaremos ResNet-18 con pesos de ImageNet y compararemos tres formas de entrenarla.
Ejecuta antes los notebooks 01 y 02 para tener los datos y los resultados de LeNet/VGG.

## Preparación

Sube la carpeta `PC3_Deep_Learning` a **Mi unidad** de Google Drive, incluyendo
`data/raw/` con las imágenes descargadas. No hace falta subir `.venv`.
Usaremos esa carpeta en los tres notebooks. Ejecútalos en orden: **01 → 02 → 03**.
Para los notebooks 02 y 03, activa una GPU en Colab.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content/drive/MyDrive/PC3_Deep_Learning
%pip install -q -r requirements.txt

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as MostrarImagen
from src.utils import seed_everything

seed_everything(42)

## Tres estrategias

| Estrategia | Capas que se entrenan | LR |
| --- | --- | --- |
| Feature extraction | Solo la capa final | 0.001 |
| Fine-tuning parcial | layer3, layer4 y capa final | 0.0001 |
| Fine-tuning total | Todas | 0.00001 |

En el ajuste parcial se congelan la entrada y los bloques layer1 y layer2.
Las estadísticas BN de las capas congeladas tampoco cambian.
Usamos imágenes de 224×224, normalización ImageNet y Adam.

In [ ]:
from src.models import build_model, parameter_counts
from src.train import DEFAULT_SETTINGS, run_suite
from src.utils import export_results

parametros = []
for estrategia in ["feature", "partial", "full"]:
    seed_everything(42)
    red = build_model("resnet18", strategy=estrategia, pretrained=True)
    parametros.append({"estrategia": estrategia, **parameter_counts(red)})
display(pd.DataFrame(parametros))

## Entrenar las tres estrategias

Usamos GPU y 5 épocas por estrategia. Las tres usan las mismas particiones,
semilla, batch y aumentos. La primera descarga de pesos necesita Internet.

In [ ]:
resultados_transfer = run_suite(DEFAULT_SETTINGS, "transfer", device="cuda")
display(resultados_transfer[["name", "trainable_parameters", "mean_train_seconds", "best_val_accuracy", "test_accuracy", "test_f1_macro"]])

## Comparar con el mejor modelo desde cero

Elegimos el mejor LeNet/VGG por su accuracy de validación. Si hay empate,
usamos la menor pérdida. No se usan los ensayos de LR alta para esta selección.
Las tablas muestran accuracy entre 0 y 1.

In [ ]:
display(export_results("results", "README.md"))
display(pd.read_csv("results/tables/transfer_vs_best_scratch.csv"))

for estrategia in ["feature", "partial", "full"]:
    display(MostrarImagen(filename=f"results/figures/resnet18_{estrategia}_seed42_curves.png"))

## Errores del mejor modelo

El mejor modelo se elige por validación. Esta matriz muestra sus errores en test.
Mira qué clases confunde más y el recall de cada clase.
`all_seven_completed` debe ser `True`: indica que terminaron los siete modelos principales.

In [ ]:
with open("results/tables/best_model_seed42.json") as archivo:
    mejor = json.load(archivo)
display(mejor)
display(MostrarImagen(filename="results/figures/best_confusion_seed42.png"))

with open("results/runs/" + mejor["name"] + "/classification_report.json") as archivo:
    reporte = json.load(archivo)
display(pd.DataFrame(reporte).T.loc[["EOSINOPHIL", "LYMPHOCYTE", "MONOCYTE", "NEUTROPHIL"]])

## Comentarios

- ¿Qué estrategia funcionó mejor y cuánto tardó?
- ¿Cuál recomendarías con pocos datos médicos? Usa accuracy, macro-F1 y recall para justificar.
- ¿Hay señales de sobreajuste?

**Respuesta:** [completar con los resultados].

Al comparar con LeNet/VGG también cambian la resolución, el optimizador y el
preentrenamiento. Entre las estrategias de ResNet cambia la LR. Por eso no podemos
atribuir toda diferencia a una sola causa. Además, faltan pruebas con pacientes o centros nuevos.

## Guardar la entrega

Los resultados y checkpoints ya quedan en Drive. Este ZIP descarga las tablas,
figuras, métricas, particiones y README, sin imágenes ni pesos del modelo.
Guarda también los tres notebooks con sus salidas.

La plantilla del informe está en el README. Completa hasta ocho páginas de contenido,
agrega las referencias y guarda el PDF como `informe.pdf`.

In [ ]:
!zip -qr /content/resultados_colab.zip results data/splits README.md -x "*.pt" "*.pth"
from google.colab import files
files.download("/content/resultados_colab.zip")